# Week 7 - Model Evaluation and Feature Engineering


- using cross validation instead of trusting a single train/test split
- doing some feature engineering (handling outliers, scaling, encoding)
- tuning hyperparameters with GridSearchCV
- comparing the "before" model (plain, untuned) with the "after" model (tuned + cross validated)


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

sns.set_style("whitegrid")

Step 1: Load the same diabetes dataset from Week 6

In [4]:
url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
data = pd.read_csv(url)
data.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## Step 2: Feature engineering

Same as last week, some columns have 0 where it should not medically be possible
(like Glucose = 0 or BMI = 0). These are basically missing values that were entered as 0.

In [5]:
# columns where 0 doesn't make sense
cols_to_fix = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

for col in cols_to_fix:
    data[col] = data[col].replace(0, np.nan)
    data[col] = data[col].fillna(data[col].median())

print("missing values left:", data.isnull().sum().sum())

missing values left: 0


In [6]:
# handling outliers using the IQR method
# anything way outside the normal range gets capped instead of removed,
# so we don't lose rows

for col in ["Insulin", "SkinThickness", "BMI"]:
    q1 = data[col].quantile(0.25)
    q3 = data[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    data[col] = data[col].clip(lower, upper)

print("outliers capped for Insulin, SkinThickness, BMI")

outliers capped for Insulin, SkinThickness, BMI


In [7]:
# a simple new feature: age group (young / middle / older)
# just to practice encoding a categorical feature

def age_group(age):
    if age < 30:
        return "young"
    elif age < 50:
        return "middle"
    else:
        return "older"

data["AgeGroup"] = data["Age"].apply(age_group)

# one-hot encode it (turns AgeGroup into 0/1 columns)
data = pd.get_dummies(data, columns=["AgeGroup"], drop_first=True)

data.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,AgeGroup_older,AgeGroup_young
0,6,148.0,72.0,35.0,125.000,33.6,0.627,50,1,True,False
1,1,85.0,66.0,29.0,125.000,26.6,0.351,31,0,False,False
2,8,183.0,64.0,29.0,125.000,23.3,0.672,32,1,False,False
3,1,89.0,66.0,23.0,112.875,28.1,0.167,21,0,False,True
4,0,137.0,40.0,35.0,135.875,43.1,2.288,33,1,False,False


Step 3: Split into features (X) and target (y), then train/test split

In [8]:
X = data.drop("Outcome", axis=1)
y = data["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# scale the features so everything is on a similar range
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("train rows:", len(X_train))
print("test rows:", len(X_test))

train rows: 614
test rows: 154


Step 4: "BEFORE" - a plain Random Forest with default settings, tested on a single split

In [9]:
before_model = RandomForestClassifier(random_state=42)
before_model.fit(X_train_scaled, y_train)

before_pred = before_model.predict(X_test_scaled)

before_acc = accuracy_score(y_test, before_pred)
before_prec = precision_score(y_test, before_pred)
before_rec = recall_score(y_test, before_pred)
before_f1 = f1_score(y_test, before_pred)

print("BEFORE (single split, default settings)")
print("accuracy :", round(before_acc, 3))
print("precision:", round(before_prec, 3))
print("recall   :", round(before_rec, 3))
print("f1 score :", round(before_f1, 3))

BEFORE (single split, default settings)
accuracy : 0.773
precision: 0.702
recall   : 0.611
f1 score : 0.653


Step 5: Why one split isn't enough - k-fold cross validation

In [10]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(before_model, X_train_scaled, y_train, cv=kfold, scoring="accuracy")

print("accuracy for each of the 5 folds:", cv_scores.round(3))
print("average accuracy across folds  :", round(cv_scores.mean(), 3))
print("standard deviation across folds:", round(cv_scores.std(), 3))
print()
print("this average is a more honest estimate of how the model performs,")
print("since it's not relying on just one lucky (or unlucky) split.")

accuracy for each of the 5 folds: [0.724 0.748 0.772 0.797 0.713]
average accuracy across folds  : 0.751
standard deviation across folds: 0.031

this average is a more honest estimate of how the model performs,
since it's not relying on just one lucky (or unlucky) split.
